# Topic 17D — Numeric Cleaning and Missing Values

Numeric columns often arrive as text, contain symbols, include impossible values, or have blanks.

**Main lesson link:** [Topic 17 — Data Cleaning with pandas](topic17_data_cleaning.ipynb)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

filename = "data_cleaning_with_pandas/data/students_messy.csv"
possible_paths = [Path(filename), Path("data") / filename, Path("../data") / filename, Path("../../data") / filename]
data_path = next(path for path in possible_paths if path.exists())
df = pd.read_csv(data_path)
print("Loaded:", data_path)
print(df.shape)


---
## 1. Inspect the Raw Numeric Column


In [ ]:
print(df["math_score"].head(15).to_string())
print("dtype:", df["math_score"].dtype)


---
## 2. Remove Symbols and Convert

`pd.to_numeric(..., errors="coerce")` converts valid numbers and turns invalid values into missing values.


In [ ]:
df["math_score_clean"] = (
    df["math_score"]
    .astype(str)
    .str.replace("%", "", regex=False)
)
df["math_score_clean"] = pd.to_numeric(df["math_score_clean"], errors="coerce")

print(df[["math_score", "math_score_clean"]].head(15))
print(df["math_score_clean"].dtype)


---
## 3. Replace Impossible Values with Missing

Scores should be between 0 and 100.


In [ ]:
impossible_mask = (df["math_score_clean"] < 0) | (df["math_score_clean"] > 100)
print("Impossible math scores:", impossible_mask.sum())
print(df.loc[impossible_mask, ["student_id", "math_score", "math_score_clean"]].head())

df.loc[impossible_mask, "math_score_clean"] = np.nan
print("After fix, impossible values:", ((df["math_score_clean"] < 0) | (df["math_score_clean"] > 100)).sum())


---
## 4. Fill Missing Values with Median

Median is often safer than mean when there are outliers.


In [ ]:
score_cols = ["math_score_clean", "english_score", "science_score"]
print("Missing before:")
print(df[score_cols].isna().sum())

for col in score_cols:
    df[col] = df[col].fillna(df[col].median())

print("\nMissing after:")
print(df[score_cols].isna().sum())


---
## 5. Check the Impact of Filling


In [ ]:
fresh = pd.read_csv(data_path)
print("English median before:", fresh["english_score"].median())
print("English mean before  :", fresh["english_score"].mean())

filled = fresh["english_score"].fillna(fresh["english_score"].median())
print("English median after :", filled.median())
print("English mean after   :", filled.mean())


### Useful Resources

- [pandas user guide: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas API: `to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html)
- [pandas API: `DataFrame.fillna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html)


---
## Quick Reflection

1. Why should impossible values become missing instead of staying in the data?
2. What does `errors="coerce"` do?
3. Why might median be safer than mean for filling scores?
